In [ ]:
import pandas as pd
import phunk
import requests
import io
import matplotlib.pyplot as plt

from socca_tune.initialize import initialize

In [ ]:
target = "Hektor"

# get data for target
r = requests.post(
    "https://api.ztf.fink-portal.org/api/v1/sso",
    json={
        "n_or_d": f"{target}",
        "withEphem": True,
        "withResiduals": True,
        "output-format": "json",
    },
)

# Format output in a DataFrame
data = pd.read_json(io.BytesIO(r.content))

In [ ]:
pc = phunk.PhaseCurve(
    target=target,
    epoch=data["Date"], # epoch of observation
    phase=data["Phase"], # Phase angle in degrees
    mag=data["i:magpsf_red"], # reduced magnitude
    mag_err=data["i:sigmapsf"], # uncertainties on magnitude
    band=data["i:fid"], # Filter IDs
)
pc.get_ephems()
pc.epoch = pc.epoch_ltc  # Light-time corrected epochs

In [ ]:
p0, metadata = initialize(pc, weights=pc.mag_err, remap=True, metadata=False)
pc.fit(models=["SOCCA"], p0=p0, weights=pc.mag_err, remap=True)

In [ ]:
import rocks

rock = rocks.Rock(target)

print(f"{'Parameter':<13} {'rocks':>12} {'SOCCA':>19}")
print("-" * 50)

rock_vals = {
    "RA": rock.parameters.physical.spin.RA0.value[0],
    "DEC": rock.parameters.physical.spin.DEC0.value[0],
    "Period": rock.parameters.physical.spin.period.value[0],
}

socca_vals = {
    "RA": pc.SOCCA.alpha,
    "DEC": pc.SOCCA.delta,
    "Period": pc.SOCCA.period * 24,
}

for key in rock_vals:
    print(f"{key:<10} {rock_vals[key]:19.6f} {socca_vals[key]:19.6f}")

In [ ]:
import numpy as np
from phunk.geometry import cos_aspect_angle, rotation_phase, subobserver_longitude
from phunk.equations import func_socca
from sbpy import photometry as phot

symbols = ["o", "s"]
filters = {1: "g", 2: "r"}
ref_jd0 = 2459580.5

# Reduced magnitude
cmred = pc.mag  # adjust names if needed

# Output arrays
m_f_g = np.full_like(pc.mag, np.nan, dtype=float)
m_f_g_s = np.full_like(pc.mag, np.nan, dtype=float)
socca_s_all = np.full_like(pc.mag, np.nan, dtype=float)
socca_g_all = np.full_like(pc.mag, np.nan, dtype=float)

# Geometry
sepB = np.degrees(
    np.arccos(
        cos_aspect_angle(
            np.radians(pc.ra),
            np.radians(pc.dec),
            np.radians(pc.SOCCA.alpha),
            np.radians(pc.SOCCA.delta),
        )
    )
)

W = rotation_phase(
    pc.epoch_ltc,
    np.radians(pc.SOCCA.W0),
    2 * np.pi / pc.SOCCA.period,
    ref_jd0,
)

sepL = subobserver_longitude(
    np.radians(pc.ra),
    np.radians(pc.dec),
    np.radians(pc.SOCCA.alpha),
    np.radians(pc.SOCCA.delta),
    W,
) % (2 * np.pi)

FONTSIZE = 18
cmap = "cividis"

fig, ax = plt.subplots(1, 3, figsize=(20, 6))
alpha = 0.5

for i, f in enumerate(pc.bands):
    cond = pc.band == f

    ax[0].scatter(
        pc.epoch_ltc[cond],
        pc.mag[cond],
        marker=symbols[i],
        alpha=alpha,
        # label=filters[f],
    )

    x_socca = [
        np.radians(pc.phase[cond]),
        np.radians(pc.ra[cond]),
        np.radians(pc.dec[cond]),
        pc.epoch_ltc[cond],
        np.radians(pc.ra_s[cond]),
        np.radians(pc.dec_s[cond]),
    ]

    H = getattr(pc.SOCCA, f"H{f}")
    G1 = getattr(pc.SOCCA, f"G1{f}")
    G2 = getattr(pc.SOCCA, f"G2{f}")

    # phase function
    socca_g = phot.HG1G2().evaluate(
        np.radians(pc.phase[cond]),
        H,
        G1,
        G2,
    )

    # shape contribution only
    socca_s = (
        func_socca(
            x_socca,
            H,
            G1,
            G2,
            np.radians(pc.SOCCA.alpha),
            np.radians(pc.SOCCA.delta),
            pc.SOCCA.period,
            pc.SOCCA.a_b,
            pc.SOCCA.a_c,
            np.radians(pc.SOCCA.W0),
        )
        - socca_g
    )

    m_f_g[cond] = cmred[cond] - socca_g
    m_f_g_s[cond] = cmred[cond] - socca_g - socca_s

    socca_g_all[cond] = socca_g
    socca_s_all[cond] = socca_s

ax[1].scatter(
    np.degrees(sepL),
    m_f_g,
    c=sepB,
    cmap=cmap,
)

idx = np.argsort(sepL)

ax[1].scatter(
    np.degrees(sepL[idx]),
    socca_s_all[idx],
    marker="*",
    c="black",
    s=10,
    label="SOCCA",
)

sc = ax[2].scatter(
    np.degrees(sepL),
    m_f_g_s,
    c=sepB,
    cmap=cmap,
)

cbar = fig.colorbar(sc)
cbar.set_label("Aspect angle (degrees)", fontsize=FONTSIZE - 2)
cbar.ax.tick_params(labelsize=FONTSIZE - 4)

ax[0].set_ylabel("Apparent magnitude $m$", fontsize=FONTSIZE)
ax[0].set_xlabel("JD", fontsize=FONTSIZE)

ax[1].set_xlabel("Sub-earth point longitude", fontsize=FONTSIZE)
ax[1].set_ylabel(r"$m - f(r, \Delta) - g(\gamma)$", fontsize=FONTSIZE)

ax[2].set_xlabel("Sub-earth point longitude", fontsize=FONTSIZE)
ax[2].set_ylabel(r"$m - f(r, \Delta) - g(\gamma)$ - SOCCA", fontsize=FONTSIZE)

for a in ax:
    a.tick_params(labelsize=FONTSIZE - 2)

ax[0].legend(fontsize=FONTSIZE - 2)
ax[1].legend(fontsize=FONTSIZE - 2)

ax[0].invert_yaxis()
fig.tight_layout()